In [10]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

In [11]:
columns = [
    "unit", "cycle", "setting_1", "setting_2", "setting_3",
    *[f"sensor_{i}" for i in range(1, 22)]
]

In [25]:
train_df = pd.read_csv(
    "/Users/macbookpro/Downloads/CMAPSSData/data/raw/train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

test_df = pd.read_csv(
    "/Users/macbookpro/Downloads/CMAPSSData/data/raw/test_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

rul_df = pd.read_csv(
    "/Users/macbookpro/Downloads/CMAPSSData/data/raw/RUL_FD001.txt",
    header=None,
    names=["RUL"]
)

In [15]:
max_cycle = train_df.groupby("unit")["cycle"].max().reset_index()
max_cycle.columns = ["unit", "max_cycle"]

In [16]:

train_df = train_df.merge(max_cycle, on="unit")
train_df["RUL"] = train_df["max_cycle"] - train_df["cycle"]
train_df = train_df.drop(columns=["max_cycle"])

In [17]:
sensor_columns = [f"sensor_{i}" for i in range(1, 22)]
setting_columns = ["setting_1", "setting_2", "setting_3"]

features = setting_columns + sensor_columns

In [18]:
X = train_df[features]
y = train_df["RUL"]

In [19]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [20]:
models = {
    "CatBoost": CatBoostRegressor(
        iterations=500,
        depth=8,
        learning_rate=0.05,
        loss_function="RMSE",
        verbose=False,
        random_seed=42
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        verbosity=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=500,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    )
}

In [21]:
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_val)

    mae = mean_absolute_error(y_val, prediction)
    rmse = np.sqrt(mean_squared_error(y_val, prediction))
    r2 = r2_score(y_val, prediction)

    results[name] = [mae, rmse, r2]

results_df = pd.DataFrame(
    results,
    index=["MAE", "RMSE", "R2"]
).T

results_df

,MAE,RMSE,R2
CatBoost,29.380313,41.073298,0.630753
LightGBM,29.897998,41.843145,0.616781
XGBoost,30.132841,42.353261,0.607380


In [22]:
def nasa_score(y_true, y_pred):
    error = y_pred - y_true
    score = np.where(
        error < 0,
        np.exp(-error / 13) - 1,
        np.exp(error / 10) - 1
    )
    return np.sum(score)

In [23]:
for name, model in models.items():
    prediction = model.predict(X_val)
    results_df.loc[name, "NASA Score"] = nasa_score(y_val.values, prediction)

results_df.sort_values("NASA Score")

,MAE,RMSE,R2,NASA Score
CatBoost,29.380313,41.073298,0.630753,3.412041e+07
LightGBM,29.897998,41.843145,0.616781,6.705379e+07
XGBoost,30.132841,42.353261,0.607380,1.930384e+08
